In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from sklearn.preprocessing import OneHotEncoder

# sparse_output=False: resultaat na encoding is normale NumPy array
encoder = OneHotEncoder(sparse_output=False)

# Load mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()


# Normalize images
train_images = train_images.astype(np.float32) / 255.0
test_images = test_images.astype(np.float32) / 255.0

train_images_flat = train_images.reshape(-1, 784)
test_images_flat = test_images.reshape(-1, 784)

# One hot encode labels
train_labels_oh = encoder.fit_transform(train_labels.reshape(-1, 1))
test_labels_oh = encoder.fit_transform(test_labels.reshape(-1, 1))


In [2]:
def relu(arr):
    return np.maximum(0, arr)

def softmax(arr):
    arr = np.array(arr, dtype=np.float64)
    shifted = arr - np.max(arr, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)

testdata = np.array([-2.0, -0.5, 0.0, 1.5, 3.0])
batch = np.array([
    [1.0, 2.0, 3.0],
    [1.0, 0.0, -1.0]
])


print(relu(testdata))
print(softmax(batch))

print("Sum of softmax outputs for each row:")
for i in range(len(batch)):
    print(f"Row {i}: {sum(softmax(batch)[i])}")


[0.  0.  0.  1.5 3. ]
[[0.09003057 0.24472847 0.66524096]
 [0.66524096 0.24472847 0.09003057]]
Sum of softmax outputs for each row:
Row 0: 0.9999999999999999
Row 1: 0.9999999999999999


In [ ]:
def forward(X, weight1, bias1, weight2, bias2):
    Z1 = X @ weight1 + bias1
    A1 = relu(Z1)
    Z2 = A1 @ weight2 + bias2
    A2 = softmax(Z2)
    cache = (X, Z1, A1, Z2, A2)
    return (A2, cache)

In [4]:
def compute_output_gradient(final_prediction, correct_answers):
    N = final_prediction.shape[0] # aantal plaatjes pet batch
    # print(f"Batch size: {N}")
    return (final_prediction - correct_answers) / N

# compute_output_gradient(pred, actual_labels)

In [5]:
def compute_output_gradients(hidden_output, output_gradient):
    dW2 = hidden_output.T @ output_gradient
    db2 = np.sum(output_gradient, axis=0)
    return (dW2, db2)

# dw2, db2 = compute_output_gradients(cache[2], compute_output_gradient(pred, actual_labels))

# print("dW2 shape:", dw2.shape)
# print("db2 shape:", db2.shape)

In [6]:
def cross_entropy(y_true, y_predicted):
    e = 1e-9 # kleine waarde om log(0) te voorkomen
    return -np.mean(np.sum(y_true * np.log(y_predicted + e), axis=1))


In [7]:
def compute_hidden_gradient(output_gradient, hidden_to_output_weights):
    return output_gradient @ hidden_to_output_weights.T

In [8]:
def relu_derivative(x):
    return (x > 0).astype(float)

In [9]:
def compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data):
    dZ1 = hidden_gradient * relu_derivative(hidden_raw_gradient)

    dW1 = input_data.T @ dZ1
    db1 = np.sum(dZ1, axis=0)
    return (dW1, db1)

In [10]:
def backward(y_true, cache, W2):
    # Tussenwaarden uit de forward pass uitpakken
    X, Z1, A1, Z2, A2 = cache

    # Fout bij de output: voorspelling - juiste antwoord (gemiddeld over de batch)
    output_gradient = compute_output_gradient(A2, y_true)

    # Gradient van output-gewichten W2 en bias b2
    dW2, db2 = compute_output_gradients(A1, output_gradient)

    # Foutsignaal terugsturen naar de hidden-laag via W2
    hidden_gradient = compute_hidden_gradient(output_gradient, W2)

    # Gradient van hidden-gewichten W1 en bias b1 (relu-afgeleide blokkeert inactieve nodes)
    dW1, db1 = compute_hidden_gradients(hidden_gradient, Z1, X)

    # Alle vier de gradienten teruggeven voor de update-stap
    return (dW1, db1, dW2, db2)

In [11]:
def normalize_and_vectorize_image(image: np.array):
    """
    Turn a image of variable dimesions (must be square) into a vector and normalize values between 0 and 1
    """
    flat = image.flatten()

    normalised: np.array = flat / 255
    return normalised

In [12]:
#Shrink
test_images_normalised_small = np.array([normalize_and_vectorize_image(np.resize(image, ((14, 14)))) for image in test_images])
print("finished test imaged")
train_images_normalised_small = np.array([normalize_and_vectorize_image(np.resize(image, (14,14))) for image in train_images])

finished test imaged


In [13]:
input_nodes_amount = 196
hidden_nodes_amount = 128
output_nodes_amount = 10

W1 = np.random.randn(input_nodes_amount, hidden_nodes_amount) * 0.01
b1 = np.zeros(hidden_nodes_amount)

W2 = np.random.randn(hidden_nodes_amount, output_nodes_amount) * 0.01
b2 = np.zeros(output_nodes_amount)

In [ ]:
lr = 0.1
batch_size = 128
n = train_images_normalised_small.shape[0]
epochs = 20

print(f"Number of training samples: {n}")

for epoch in range(epochs):

    # train in batches van batch_size
    for start in range(0, n, batch_size):
        b = slice(start, start + batch_size)
        voorspelling, cache = forward(train_images_normalised_small[b], W1, b1, W2, b2)
        dW1, db1, dW2, db2 = backward(train_labels_oh[b], cache, W2)
        W1 -= lr*dW1
        b1 -= lr*db1
        W2 -= lr*dW2
        b2 -= lr*db2

    # einde epoch: loss/acc op de hele set
    pred_all, _ = forward(train_images_normalised_small, W1, b1, W2, b2)
    loss = cross_entropy(train_labels_oh, pred_all)
    acc = np.mean(np.argmax(pred_all, axis=1) == train_labels)
    print(f"Epoch {epoch + 1}, Loss: {loss:.4f}, Acc: {acc:.3f}")

Number of training samples: 60000
Epoch 1, Loss: 2.3013, Acc: 0.112
Epoch 2, Loss: 2.3013, Acc: 0.112
Epoch 3, Loss: 2.3013, Acc: 0.112
Epoch 4, Loss: 2.3013, Acc: 0.112
Epoch 5, Loss: 2.3013, Acc: 0.112
Epoch 6, Loss: 2.3013, Acc: 0.112
Epoch 7, Loss: 2.3013, Acc: 0.112
Epoch 8, Loss: 2.3013, Acc: 0.112
Epoch 9, Loss: 2.3013, Acc: 0.112
Epoch 10, Loss: 2.3013, Acc: 0.112
Epoch 11, Loss: 2.3013, Acc: 0.112


KeyboardInterrupt: 

In [ ]:
# Export weights
destination = input("Desitination path")
np.save(destination + "W1.npy", W1, True)
np.save(destination + "b1.npy", b1, True)
np.save(destination + "W2.npy", W2, True)
np.save(destination + "b2.npy", b2, True)